# Лабораторная работа. Структуры данных pandas и индексация

**Цель работы:** освоить основные структуры данных библиотеки pandas (`Series`, `DataFrame`, `Index`), научиться загружать данные из файлов, исследовать их, выбирать, фильтровать, сортировать и преобразовывать, а также эффективно работать с большими наборами данных.

**Используемые данные:**
- `sp500.csv` — сведения о 500 компаниях индекса S&P 500 (небольшой CSV, ~80 КБ);
- `sales_1m.csv` — синтетический набор продаж на 1 000 000 строк (генерируется в разделе 13).

**Содержание**
1. Подготовка к работе
2. Структуры данных: Series и DataFrame
3. Объект Index
4. Чтение данных из файлов
5. Первичный осмотр данных
6. Доступ к данным: столбцы, loc, iloc, at, iat, срезы
7. Копирование и ссылки
8. Фильтрация: логические маски, isin(), query()
9. Сортировка
10. Изменение структуры: создание, переименование, удаление столбцов
11. Арифметические операции и выравнивание данных
12. Индексация подробнее: set_index, reindex, MultiIndex
13. Работа с большим датасетом: память, типы, usecols, dtype, chunksize
14. Задания для самостоятельной работы
15. Контрольные вопросы

## 1. Подготовка к работе

**pandas** — библиотека Python для анализа табличных данных. Она построена поверх **NumPy**: значения столбцов хранятся в массивах NumPy (или в расширенных типах pandas), поэтому операции над целыми столбцами выполняются быстро, без циклов Python (векторизация).

Традиционные сокращения при импорте: `pd` для pandas и `np` для NumPy.

In [9]:
import os
import time
import numpy as np
import pandas as pd

print('pandas', pd.__version__)
print('numpy ', np.__version__)

pandas 3.0.5
numpy  2.3.5


Основной набор данных — `sp500.csv` (500 компаний индекса S&P 500, 15 столбцов). Положите файл в папку `../data/` или рядом с notebook.

| Column Name | Description |
| ------------- |:-------------:|
|Symbol|Сокращенное название организации|
|Name|Полное название организации|
|Sector|Сектор экономики|
|Price|Стоимость акции|
|Dividend Yield|Дивидендная доходность|
|Price/Earnings|Цена / прибыль|
|Earnings/Share|Прибыль на акцию|
|Book Value|Балансовая стоимость компании|
|52 week low|52-недельный минимум|
|52 week high|52-недельный максимум|
|Market Cap|Рыночная капитализация|
|EBITDA|**E**arnings **b**efore **i**nterest, **t**axes, **d**epreciation and **a**mortization|
|Price/Sales|Цена / объём продаж|
|Price/Book|Цена / балансовая стоимость|
|SEC Filings|Ссылка *sec.gov*|

In [10]:
# ищем файл в типичных местах
for path in ['../data/sp500.csv', 'sp500.csv', 'data/sp500.csv']:
    if os.path.exists(path):
        DATA_PATH = path
        break
else:
    raise FileNotFoundError('Положите sp500.csv в ../data/ или рядом с notebook')
print('Используется файл:', DATA_PATH)

Используется файл: data/sp500.csv


## 2. Структуры данных: Series и DataFrame

### Теория
- **Series** — одномерный размеченный массив: значения (`values`) одного типа + метки (`index`). Похож на словарь «метка → значение» и одновременно на массив NumPy.
- **DataFrame** — двумерная таблица: набор столбцов-Series с общим индексом строк. У DataFrame два индекса: `index` (строки) и `columns` (столбцы). Разные столбцы могут иметь разные типы.
- Если индекс не задан, pandas создаёт `RangeIndex` 0, 1, 2, …

Основные конструкторы:
```python
pd.Series(data, index=None, dtype=None, name=None)
pd.DataFrame(data, index=None, columns=None, dtype=None)
```
`data` может быть списком, словарём, массивом NumPy, скаляром, другим Series/DataFrame.

### 2.1 Создание из списков

создаем Series значений из списка целых чисел с явным индексом

In [11]:
s = pd.Series(data=[10, 11, 12, 13, 14],
              index=[1, 2, 3, 5, 7])
s

1    10
2    11
3    12
5    13
7    14
dtype: int64

создаем Series из строковых значений (тип `object`/`str`), индекс создаётся автоматически

In [12]:
s = pd.Series(['Blue', 'Yellow', 'Green'])
s

0      Blue
1    Yellow
2     Green
dtype: str

создаем DataFrame из двумерного списка (список строк) и задаём имена столбцов

In [13]:
df = pd.DataFrame([[10, 11], [20, 21], [30, 31]],
                  columns=['A', 'B'])
df

,A,B
0,10,11
1,20,21
2,30,31


создаем DataFrame из списка объектов Series: каждая Series становится **строкой**

In [14]:
series_1 = pd.Series([70, 90])
series_2 = pd.Series([71, 91])
df = pd.DataFrame([series_1, series_2])
df

,0,1
0,70,90
1,71,91


имена столбцов можно задать и после создания датафрейма

In [15]:
df.columns = ['col_1', 'col_2']
df

,col_1,col_2
0,70,90
1,71,91


### 2.2 Создание из словарей
- Для **Series** ключи словаря становятся индексом, значения — данными.
- Для **DataFrame** ключи становятся **именами столбцов**, а значения (списки или Series) — содержимым столбцов.

In [16]:
s = pd.Series({'Homer': 'Dad',
               'Marge': 'Mom',
               'Bart': 'Son',
               'Lisa': 'Daughter',
               'Maggie': 'Daughter'})
s

Homer          Dad
Marge          Mom
Bart           Son
Lisa      Daughter
Maggie    Daughter
dtype: str

DataFrame из словаря списков

In [17]:
temperatures = {'col_1': [70, 71],
                'col_2': [90, 91]}
pd.DataFrame(temperatures)

,col_1,col_2
0,70,90
1,71,91


DataFrame из словаря Series: каждая Series становится **столбцом**

In [18]:
series_1 = pd.Series([70, 71])
series_2 = pd.Series([90, 91])
df = pd.DataFrame({'col_1': series_1,
                   'col_2': series_2})
df

,col_1,col_2
0,70,90
1,71,91


### 2.3 Создание из массивов NumPy
Часто данные генерируют функциями NumPy:
- `np.arange(start, stop, step)` — числа от **start** до **stop-1** с шагом **step**;
- `np.linspace(start, stop, num)` — `num` равномерно распределённых точек на отрезке, включая концы;
- `np.random.*` — случайные числа. Фиксация `seed` делает результаты воспроизводимыми.

In [19]:
pd.Series(np.arange(15, 25, 2))

0    15
1    17
2    19
3    21
4    23
dtype: int64

In [20]:
pd.Series(np.linspace(0, 9, 5))

0    0.00
1    2.25
2    4.50
3    6.75
4    9.00
dtype: float64

Series из 5 нормально распределенных случайных чисел

In [21]:
np.random.seed(123)
pd.Series(np.random.normal(size=5))

0   -1.085631
1    0.997345
2    0.282978
3   -1.506295
4   -0.578600
dtype: float64

DataFrame 4×3 из двумерного массива NumPy с заданными индексом и столбцами

In [22]:
np.random.seed(123)
df = pd.DataFrame(np.random.normal(size=12).reshape(4, 3),
                  index=['ind_1', 'ind_2', 'ind_3', 'ind_4'],
                  columns=['col_1', 'col_2', 'col_3'])
df

,col_1,col_2,col_3
ind_1,-1.085631,0.997345,0.282978
ind_2,-1.506295,-0.578600,1.651437
ind_3,-2.426679,-0.428913,1.265936
ind_4,-0.866740,-0.678886,-0.094709


### 2.4 Объекты для дальнейших примеров
`Simpsons` — Series с текстовым индексом, `numbers` — Series с целочисленным индексом 25…34 (не совпадающим с позициями 0…9, это важно для понимания разницы `loc` и `iloc`).

In [23]:
Simpsons = pd.Series({'Homer': 120, 'Marge': 60, 'Bart': 35, 'Lisa': 30, 'Maggie': 7})
Simpsons

Homer     120
Marge      60
Bart       35
Lisa       30
Maggie      7
dtype: int64

In [24]:
np.random.seed(123)
numbers = pd.Series(data=np.random.normal(size=10),
                    index=np.arange(25, 35))
numbers

25   -1.085631
26    0.997345
27    0.282978
28   -1.506295
29   -0.578600
30    1.651437
31   -2.426679
32   -0.428913
33    1.265936
34   -0.866740
dtype: float64

## 3. Объект Index

### Теория
**Index** — это объект, хранящий метки строк (`df.index`) или столбцов (`df.columns`). Его свойства:
- **неизменяемость** — нельзя присвоить значение отдельному элементу (`idx[0] = ...` вызовет ошибку), можно только заменить индекс целиком;
- может содержать **повторяющиеся** метки;
- поддерживает **операции над множествами**: `union`, `intersection`, `difference`, `symmetric_difference`;
- обеспечивает **быстрый поиск** по меткам (хеш-таблица), см. раздел 12.

Основные виды индексов:
| Тип | Когда возникает |
|---|---|
| `RangeIndex` | индекс по умолчанию 0, 1, 2, … (хранит только start/stop/step, почти не занимает памяти) |
| `Index` с `dtype=int64` / `float64` / `object` | метки — целые / дробные числа / строки |
| `DatetimeIndex` | метки — даты и время |
| `CategoricalIndex` | метки — категории |
| `MultiIndex` | многоуровневый (иерархический) индекс |

> В pandas 2.0+ классы `Int64Index` и `Float64Index` удалены: вместо них используется обычный `Index` с соответствующим `dtype`.

In [25]:
idx = pd.Index(['a', 'b', 'c', 'd'], name='letters')
print(idx)
print('тип меток:', idx.dtype, '| размер:', idx.size)
try:
    idx[0] = 'z'
except TypeError as e:
    print('Index неизменяем:', e)

Index(['a', 'b', 'c', 'd'], dtype='str', name='letters')
тип меток: str | размер: 4
Index неизменяем: Index does not support mutable operations


операции над множествами

In [26]:
idx2 = pd.Index(['c', 'd', 'e'])
print('объединение:', idx.union(idx2))
print('пересечение:', idx.intersection(idx2))
print('разность:   ', idx.difference(idx2))

объединение: Index(['a', 'b', 'c', 'd', 'e'], dtype='str')
пересечение: Index(['c', 'd'], dtype='str')
разность:    Index(['a', 'b'], dtype='str')


типы индексов: `RangeIndex` по умолчанию, целочисленный и дробный индекс

In [27]:
df_range = pd.DataFrame(np.arange(10, 15))
df_i64 = pd.DataFrame(np.arange(10, 20), index=np.arange(0, 10))
df_f64 = pd.DataFrame(np.arange(0, 1000, 5), index=np.arange(0.0, 100.0, 0.5))
print(df_range.index)
print(df_i64.index)
print(df_f64.index[:5])

RangeIndex(start=0, stop=5, step=1)
Index([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype='int64')
Index([0.0, 0.5, 1.0, 1.5, 2.0], dtype='float64')


у Series и DataFrame индекс доступен через `.index`, значения — через `.values`; столбцы DataFrame — тоже объект Index

In [28]:
print(Simpsons.index)
print(Simpsons.values)
print(df.columns)

Index(['Homer', 'Marge', 'Bart', 'Lisa', 'Maggie'], dtype='str')
[120  60  35  30   7]
Index(['col_1', 'col_2', 'col_3'], dtype='str')


индексу и самой Series можно дать имя

In [29]:
Simpsons.name = 'Simpsons weight'
Simpsons.index.name = 'First name'
Simpsons

First name
Homer     120
Marge      60
Bart       35
Lisa       30
Maggie      7
Name: Simpsons weight, dtype: int64

## 4. Чтение данных из файлов

### Теория
pandas читает данные из множества форматов функциями `pd.read_*` и записывает методами `df.to_*`:

| Формат | Чтение | Запись |
|---|---|---|
| CSV / текст | `pd.read_csv()` | `df.to_csv()` |
| Excel | `pd.read_excel()` (нужен пакет `openpyxl`) | `df.to_excel()` |
| JSON | `pd.read_json()` | `df.to_json()` |
| Parquet, SQL, HTML, … | `read_parquet`, `read_sql`, `read_html` | `to_parquet`, `to_sql`, … |

Важные параметры `read_csv()`:
- `sep` — разделитель (`,`, `;`, `\t`);
- `nrows` — прочитать только первые N строк (удобно для «разведки» большого файла);
- `usecols` — прочитать только нужные столбцы;
- `index_col` — столбец, который станет индексом;
- `dtype` — типы столбцов (подробнее в разделе 13);
- `chunksize` — читать по частям, возвращая итератор;
- `encoding`, `decimal`, `parse_dates`, `na_values`- Кодировка текста файла (по умолчанию utf-8), Символ десятичного разделителя (по умолчанию "."), Какие столбцы преобразовать в даты/время, Дополнительные строки, которые считать NaN  соответственно и др.

[Документация по вводу-выводу](http://pandas.pydata.org/pandas-docs/stable/user_guide/io.html)

### 4.1 read_csv()

неверный разделитель: вся строка попадает в один столбец

In [30]:
pd.read_csv(DATA_PATH, sep=';').head(3)

,"Symbol,Name,Sector,Price,Dividend Yield,Price/Earnings,Earnings/Share,Book Value,52 week low,52 week high,Market Cap,EBITDA,Price/Sales,Price/Book,SEC Filings"
0,"MMM,3M Co.,Industrials,141.14,2.12,20.33,6.90,..."
1,"ABT,Abbott Laboratories,Health Care,39.60,1.82..."
2,"ABBV,AbbVie Inc.,Health Care,53.95,3.02,20.87,..."


правильный разделитель + только 3 строки (`nrows`)

In [31]:
pd.read_csv(DATA_PATH, sep=',', nrows=3)

,Symbol,Name,Sector,Price,Dividend Yield,Price/Earnings,Earnings/Share,Book Value,52 week low,52 week high,Market Cap,EBITDA,Price/Sales,Price/Book,SEC Filings
0,MMM,3M Co.,Industrials,141.14,2.12,20.33,6.900,26.668,107.15,143.37,92.345,8.121,2.95,5.26,http://www.sec.gov/cgi-bin/browse-edgar?action...
1,ABT,Abbott Laboratories,Health Care,39.60,1.82,25.93,1.529,15.573,32.70,40.49,59.477,4.359,2.74,2.55,http://www.sec.gov/cgi-bin/browse-edgar?action...
2,ABBV,AbbVie Inc.,Health Care,53.95,3.02,20.87,2.570,2.954,40.10,54.78,85.784,7.190,4.48,18.16,http://www.sec.gov/cgi-bin/browse-edgar?action...


только нужные столбцы (`usecols`) и столбец `Symbol` в качестве индекса (`index_col`)

In [32]:
pd.read_csv(DATA_PATH, nrows=3,
            usecols=['Symbol', 'Sector', 'Price', 'Book Value'],
            index_col='Symbol')

,Sector,Price,Book Value
Symbol,,,
MMM,Industrials,141.14,26.668
ABT,Health Care,39.60,15.573
ABBV,Health Care,53.95,2.954


`chunksize` возвращает итератор: файл читается порциями по 50 строк

In [33]:
df_chunk = pd.read_csv(DATA_PATH, chunksize=50,
                       usecols=['Symbol', 'Sector', 'Price', 'Book Value'],
                       index_col='Symbol')
for df_tmp in df_chunk:
    print('DataFrame part:', df_tmp.shape)

DataFrame part: (50, 3)
DataFrame part: (50, 3)
DataFrame part: (50, 3)
DataFrame part: (50, 3)
DataFrame part: (50, 3)
DataFrame part: (50, 3)
DataFrame part: (50, 3)
DataFrame part: (50, 3)
DataFrame part: (50, 3)
DataFrame part: (50, 3)


загружаем основной датафрейм `sp500`, с которым будем работать дальше

In [34]:
sp500 = pd.read_csv(DATA_PATH,
                    usecols=['Symbol', 'Sector', 'Price', 'Book Value'],
                    index_col='Symbol')
sp500

,Sector,Price,Book Value
Symbol,,,
MMM,Industrials,141.14,26.668
ABT,Health Care,39.60,15.573
ABBV,Health Care,53.95,2.954
ACN,Information Technology,79.79,8.326
ACE,Financials,102.91,86.897
...,...,...,...
YHOO,Information Technology,35.02,12.768
YUM,Consumer Discretionary,74.77,5.147
ZMH,Health Care,101.84,37.181


### 4.2 read_excel() и read_json()
Сохраним часть данных в Excel и JSON и прочитаем обратно. Для JSON параметр `orient` задаёт структуру файла (`records` — список объектов-строк, `columns`, `index`, `split` и др.).

In [35]:
sample = sp500.head(10).reset_index()
# Сохраняем DataFrame в файл Excel:
# index=False — не записывать индекс строк как отдельный столбец;
# sheet_name='data' — задать имя листа в книге Excel.
sample.to_excel('sp500_sample.xlsx', index=False, sheet_name='data')

# Сохраняем DataFrame в JSON:
# orient='records' — каждая строка таблицы станет отдельным объектом
# вида {"столбец": значение, ...};
# force_ascii=False — разрешаем не-ASCII символы (например, кириллицу)
# записывать как есть, а не в виде escape-последовательностей \uXXXX.
sample.to_json('sp500_sample.json', orient='records', force_ascii=False)

df_xlsx = pd.read_excel('sp500_sample.xlsx', sheet_name='data', index_col='Symbol')
df_json = pd.read_json('sp500_sample.json', orient='records').set_index('Symbol')
display(df_xlsx.head(3), df_json.head(3))
print('Данные совпадают:', df_xlsx.equals(df_json))

,Sector,Price,Book Value
Symbol,,,
MMM,Industrials,141.14,26.668
ABT,Health Care,39.60,15.573
ABBV,Health Care,53.95,2.954


,Sector,Price,Book Value
Symbol,,,
MMM,Industrials,141.14,26.668
ABT,Health Care,39.60,15.573
ABBV,Health Care,53.95,2.954


Данные совпадают: True


так выглядит JSON в формате `records`:

In [36]:
print(open('sp500_sample.json', encoding='utf-8').read()[:250])

[{"Symbol":"MMM","Sector":"Industrials","Price":141.14,"Book Value":26.668},{"Symbol":"ABT","Sector":"Health Care","Price":39.6,"Book Value":15.573},{"Symbol":"ABBV","Sector":"Health Care","Price":53.95,"Book Value":2.954},{"Symbol":"ACN","Sector":"I


## 5. Первичный осмотр данных

### Теория
После загрузки данных первым делом нужно понять их структуру:

| Атрибут / метод | Что показывает |
|---|---|
| `.head(n)`, `.tail(n)` | первые / последние n строк (по умолчанию 5) |
| `.shape` | кортеж (число строк, число столбцов) |
| `len(df)`, `.size` | число строк; общее число ячеек |
| `.columns`, `.index` | метки столбцов и строк |
| `.dtypes` | тип каждого столбца |
| `.info()` | сводка: индекс, столбцы, число непустых значений, типы, память |
| `.describe()` | описательные статистики (count, mean, std, min, квартили, max) |
| `.nunique()` | число уникальных значений |

Атрибуты (`shape`, `columns`, `dtypes`) пишутся **без скобок**, методы (`head()`, `info()`) — **со скобками**.

In [37]:
sp500.head()

,Sector,Price,Book Value
Symbol,,,
MMM,Industrials,141.14,26.668
ABT,Health Care,39.60,15.573
ABBV,Health Care,53.95,2.954
ACN,Information Technology,79.79,8.326
ACE,Financials,102.91,86.897


In [38]:
sp500.tail(3)

,Sector,Price,Book Value
Symbol,,,
ZMH,Health Care,101.84,37.181
ZION,Financials,28.43,30.191
ZTS,Health Care,30.53,2.150


In [39]:
print('shape:', sp500.shape)
print('len  :', len(sp500), '| size:', sp500.size)
print('columns:', sp500.columns.tolist())

shape: (500, 3)
len  : 500 | size: 1500
columns: ['Sector', 'Price', 'Book Value']


In [40]:
sp500.dtypes

Sector            str
Price         float64
Book Value    float64
dtype: object

In [41]:
sp500.info()

<class 'pandas.DataFrame'>
Index: 500 entries, MMM to ZTS
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Sector      500 non-null    str    
 1   Price       500 non-null    float64
 2   Book Value  499 non-null    float64
dtypes: float64(2), str(1)
memory usage: 15.6+ KB


для числовых столбцов — статистики распределения, для текстовых (`exclude='number'` — все нечисловые столбцы) — count, unique, top, freq

In [42]:
sp500.describe()

,Price,Book Value
count,500.00000,499.000000
mean,77.36860,301.611301
std,87.59694,6195.727862
min,0.00000,-51.275000
25%,38.74500,10.836500
50%,58.35500,19.098000
75%,86.68000,31.386500
max,1197.12000,138425.453100


In [43]:
sp500.describe(exclude='number')

,Sector
count,500
unique,13
top,Consumer Discretionary
freq,85


In [44]:
sp500.nunique()

Sector         13
Price         495
Book Value    495
dtype: int64

`nunique()` показывает 13 секторов, хотя в S&P 500 их 10. Посмотрим на значения внимательнее: `repr` показывает лишние пробелы, а `Industries` — опечатка вместо `Industrials`. Это типичная проблема «грязных» данных, её нужно замечать на этапе осмотра.

In [45]:
print([repr(x) for x in sp500['Sector'].unique()])
sp500['Sector'].value_counts()  # частота встречаемости

["'Industrials'", "'Health Care'", "'Information Technology'", "'Financials'", "'Utilities'", "'Materials'", "'Consumer Staples'", "'Consumer Discretionary'", "'Energy'", "'Telecommunications Services'", "'Consumer Staples '", "'Industries'", "'Consumer Discretionary '"]


Sector
Consumer Discretionary         85
Financials                     82
Industrials                    64
Information Technology         64
Health Care                    54
Energy                         41
Consumer Staples               39
Utilities                      33
Materials                      29
Telecommunications Services     6
Consumer Staples                1
Industries                      1
Consumer Discretionary          1
Name: count, dtype: int64

исправим: уберём пробелы по краям и заменим опечатку

In [46]:
sp500['Sector'] = sp500['Sector'].str.strip().replace({'Industries': 'Industrials'})
print('секторов после очистки:', sp500['Sector'].nunique())

секторов после очистки: 10


### 5.1 Настройки вывода
pandas сокращает вывод больших таблиц. Число отображаемых строк задаёт `pd.options.display.max_rows`. [Все настройки](http://pandas.pydata.org/pandas-docs/stable/user_guide/options.html#available-options)

In [47]:
print('по умолчанию:', pd.options.display.max_rows)
pd.options.display.max_rows = 10
sp500

по умолчанию: 60


,Sector,Price,Book Value
Symbol,,,
MMM,Industrials,141.14,26.668
ABT,Health Care,39.60,15.573
ABBV,Health Care,53.95,2.954
ACN,Information Technology,79.79,8.326
ACE,Financials,102.91,86.897
...,...,...,...
YHOO,Information Technology,35.02,12.768
YUM,Consumer Discretionary,74.77,5.147
ZMH,Health Care,101.84,37.181


## 6. Доступ к данным

### Теория
| Способ | Принцип | Пример | Результат |
|---|---|---|---|
| `df['col']` | столбец по имени | `sp500['Price']` | Series |
| `df[['c1','c2']]` | несколько столбцов | `sp500[['Price','Sector']]` | DataFrame |
| `df.col` | атрибутивный доступ (только для имён без пробелов) | `sp500.Price` | Series |
| `.loc[строки, столбцы]` | по **меткам** | `sp500.loc['MMM', 'Price']` | скаляр/Series/DataFrame |
| `.iloc[строки, столбцы]` | по **позициям** (с 0) | `sp500.iloc[0, 1]` | скаляр/Series/DataFrame |
| `.at[метка, столбец]` | одно значение по меткам (быстрее `loc`) | `sp500.at['MMM','Price']` | скаляр |
| `.iat[i, j]` | одно значение по позициям (быстрее `iloc`) | `sp500.iat[0, 1]` | скаляр |

Правило: одна метка → Series (или скаляр), **список** меток → DataFrame.

Срезы: в `iloc` правая граница **не включается** (как в Python), в `loc` правая граница **включается**.

### 6.1 Столбцы

In [48]:
print(type(sp500['Sector']))
sp500['Sector'].head()

<class 'pandas.Series'>


Symbol
MMM                Industrials
ABT                Health Care
ABBV               Health Care
ACN     Information Technology
ACE                 Financials
Name: Sector, dtype: str

In [49]:
print(type(sp500[['Price', 'Book Value']]))
sp500[['Price', 'Book Value']].head()

<class 'pandas.DataFrame'>


,Price,Book Value
Symbol,,
MMM,141.14,26.668
ABT,39.60,15.573
ABBV,53.95,2.954
ACN,79.79,8.326
ACE,102.91,86.897


атрибутивный доступ работает только для имён-идентификаторов: `sp500.Book Value` — синтаксическая ошибка, поэтому используем квадратные скобки

In [50]:
print(sp500.Price.head(3))
# sp500.Book Value          -> SyntaxError
print(sp500['Book Value'].head(3))

Symbol
MMM     141.14
ABT      39.60
ABBV     53.95
Name: Price, dtype: float64
Symbol
MMM     26.668
ABT     15.573
ABBV     2.954
Name: Book Value, dtype: float64


### 6.2 Строки по метке: loc

**Series**: метки 25 и 33 (а не позиции!); метки 0 в индексе `numbers` нет — будет ошибка `KeyError`

In [51]:
print(numbers.loc[[25, 33]])
try:
    numbers.loc[0]
except KeyError as e:
    print('KeyError: нет метки', e)

25   -1.085631
33    1.265936
dtype: float64
KeyError: нет метки 0


**DataFrame**: одна метка → Series, список меток → DataFrame

In [52]:
print((sp500.loc['MMM']))
sp500.loc['MMM']

Sector        Industrials
Price              141.14
Book Value         26.668
Name: MMM, dtype: object


Sector        Industrials
Price              141.14
Book Value         26.668
Name: MMM, dtype: object

In [53]:
sp500.loc[['MMM', 'MSFT']]

,Sector,Price,Book Value
Symbol,,,
MMM,Industrials,141.14,26.668
MSFT,Information Technology,40.12,10.584


### 6.3 Строки по позиции: iloc

**Series**: позиции 5 и −5 (пятая с конца); позиции 10 нет — ошибка `IndexError`

In [54]:
print(numbers.iloc[[5, -5]])
try:
    numbers.iloc[10]
except IndexError as e:
    print('IndexError:', e)

30    1.651437
30    1.651437
dtype: float64
IndexError: single positional indexer is out-of-bounds


**DataFrame**: строки на позициях 0 и 2; позицию метки можно узнать через `index.get_loc()`

In [55]:
sp500.iloc[[0, 2]]

,Sector,Price,Book Value
Symbol,,,
MMM,Industrials,141.14,26.668
ABBV,Health Care,53.95,2.954


In [56]:
i1, i2 = sp500.index.get_loc('MMM'), sp500.index.get_loc('A')
print(i1, i2)
sp500.iloc[[i1, i2]]

0 10


,Sector,Price,Book Value
Symbol,,,
MMM,Industrials,141.14,26.668
A,Health Care,56.18,16.928


### 6.4 Скалярные значения: at, iat

In [57]:
print('at :', sp500.at['MMM', 'Price'])
print('iat:', sp500.iat[0, 1])

at : 141.14
iat: 141.14


### 6.5 Одновременный отбор строк и столбцов

предпочтительно указывать строки и столбцы в **одном** вызове `loc` / `iloc` (а не цепочкой `[...][...]`)

In [58]:
sp500.loc[['ABT', 'ZTS'], ['Sector', 'Price']]

,Sector,Price
Symbol,,
ABT,Health Care,39.60
ZTS,Health Care,30.53


In [59]:
print(sp500.index.get_loc('ABT'), sp500.index.get_loc('ZTS'))
sp500.iloc[[1, 499], [0, 1]]

1 499


,Sector,Price
Symbol,,
ABT,Health Care,39.60
ZTS,Health Care,30.53


### 6.6 Срезы
Срез задаётся правилом `[начало : конец : шаг]`:
- в `iloc` правая граница **не включается**, в `loc` — **включается**;
- шаг и позиция могут быть отрицательными (отсчёт с конца);
- нумерация позиций — с нуля.

In [60]:
numbers.iloc[1:6]     # позиции 1..5

26    0.997345
27    0.282978
28   -1.506295
29   -0.578600
30    1.651437
dtype: float64

In [61]:
numbers.iloc[1:6:2]   # позиции 1, 3, 5

26    0.997345
28   -1.506295
30    1.651437
dtype: float64

In [62]:
numbers.iloc[:6]

25   -1.085631
26    0.997345
27    0.282978
28   -1.506295
29   -0.578600
30    1.651437
dtype: float64

In [63]:
numbers.iloc[3:]

28   -1.506295
29   -0.578600
30    1.651437
31   -2.426679
32   -0.428913
33    1.265936
34   -0.866740
dtype: float64

In [64]:
numbers.iloc[5::-1]   # в обратном порядке, начиная с позиции 5

30    1.651437
29   -0.578600
28   -1.506295
27    0.282978
26    0.997345
25   -1.085631
dtype: float64

In [65]:
numbers.iloc[-4:]     # 4 последних

31   -2.426679
32   -0.428913
33    1.265936
34   -0.866740
dtype: float64

DataFrame: срезы по позициям и по меткам

In [66]:
sp500.iloc[4::-1]

,Sector,Price,Book Value
Symbol,,,
ACE,Financials,102.91,86.897
ACN,Information Technology,79.79,8.326
ABBV,Health Care,53.95,2.954
ABT,Health Care,39.60,15.573
MMM,Industrials,141.14,26.668


In [67]:
sp500.loc['ABT':'ACN']   # правая метка включена

,Sector,Price,Book Value
Symbol,,,
ABT,Health Care,39.60,15.573
ABBV,Health Care,53.95,2.954
ACN,Information Technology,79.79,8.326


### 6.7 Случайная подвыборка и транспонирование

In [68]:
sp500.sample(n=3, random_state=1)

,Sector,Price,Book Value
Symbol,,,
TAP,Consumer Staples,64.40,46.542
OKE,Utilities,64.81,2.812
AN,Consumer Discretionary,56.38,17.327


In [69]:
sp500.sample(frac=0.01, replace=True, random_state=777)   # с возвращением

,Sector,Price,Book Value
Symbol,,,
CLF,Materials,16.34,34.523
MHK,Consumer Discretionary,136.41,61.582
BDX,Health Care,115.70,27.510
HOT,Consumer Discretionary,78.73,17.218
CELG,Health Care,150.13,11.200


In [70]:
sp500.head().T

Symbol,MMM,ABT,ABBV,ACN,ACE
Sector,Industrials,Health Care,Health Care,Information Technology,Financials
Price,141.14,39.6,53.95,79.79,102.91
Book Value,26.668,15.573,2.954,8.326,86.897


## 7. Копирование и ссылки

### Теория
Выборка данных может быть **копией** (новые данные) или **представлением** (view — ссылкой на исходные данные). Раньше изменение выборки иногда меняло исходный объект и вызывало предупреждение `SettingWithCopyWarning`.

Начиная с **pandas 3.0** по умолчанию действует режим **Copy-on-Write**: любой результат выборки ведёт себя как копия. Изменения выборки **никогда не меняют исходный объект**, а данные физически копируются только в момент изменения.

Практические правила:
- чтобы изменить исходный датафрейм, присваивайте через `df.loc[строки, столбец] = значение`;
- если выборку планируется изменять и хранить, явно вызывайте `.copy()`;
- не используйте цепочечное присваивание `df[...][...] = ...` — в pandas 3 оно не изменяет `df`.

In [71]:
n = numbers.iloc[1:5]       # выборка
n.loc[:] = 0                # изменяем выборку
print(n)
print('\nИсходный numbers не изменился:\n', numbers.iloc[:6])

26    0.0
27    0.0
28    0.0
29    0.0
dtype: float64

Исходный numbers не изменился:
 25   -1.085631
26    0.997345
27    0.282978
28   -1.506295
29   -0.578600
30    1.651437
dtype: float64


изменение исходного объекта — только через `loc`/`iloc` на нём самом; для экспериментов работаем с копией

In [72]:
numbers_copy = numbers.copy()
numbers_copy.iloc[1:5] = 0
display(pd.DataFrame({'numbers': numbers, 'numbers_copy': numbers_copy}).head(6))

,numbers,numbers_copy
25,-1.085631,-1.085631
26,0.997345,0.000000
27,0.282978,0.000000
28,-1.506295,0.000000
29,-0.578600,0.000000
30,1.651437,1.651437


## 8. Фильтрация: логические маски, isin(), query()

### Теория
**Логическая маска** — Series из `True`/`False` той же длины, что и данные. Передав её в `[]` или `loc`, получаем только строки с `True`.

- Сравнения: `==`, `!=`, `>`, `<`, `>=`, `<=`.
- Логические операторы: `&` (И), `|` (ИЛИ), `~` (НЕ). Ключевые слова `and`/`or`/`not` для Series **не работают**.
- **Каждое условие — в скобках**: `(x > 0) & (x < 1)`. Без скобок `x > 0 & x < 1` вызовет ошибку из‑за приоритета операторов.
- `.isin(список)` — проверка принадлежности множеству значений (замена цепочки `==` через `|`).
- `.query("строка условия")` — компактная запись фильтра; внешние переменные подставляются через `@имя`.
- `.where(условие, other)` — сохраняет форму, заменяя неподходящие значения на `NaN` или `other`.
- `.all()`, `.any()`, `.sum()` для масок: все ли / хотя бы одно / сколько `True`.

### 8.1 Series

In [73]:
logical_results = (numbers > 0) & (numbers < 1)
logical_results

25    False
26     True
27     True
28    False
29    False
30    False
31    False
32    False
33    False
34    False
dtype: bool

In [74]:
numbers[logical_results]

26    0.997345
27    0.282978
dtype: float64

In [75]:
numbers.where((numbers > 0) & (numbers < 1), other=-1)

25   -1.000000
26    0.997345
27    0.282978
28   -1.000000
29   -1.000000
30   -1.000000
31   -1.000000
32   -1.000000
33   -1.000000
34   -1.000000
dtype: float64

In [76]:
print('все >= 0 :', (numbers >= 0).all())
print('есть < 2 :', (numbers < 2).any())
print('сколько < 1:', (numbers < 1).sum())

все >= 0 : False
есть < 2 : True
сколько < 1: 8


### 8.2 DataFrame

In [77]:
sp500[sp500.Price < 100].head()

,Sector,Price,Book Value
Symbol,,,
ABT,Health Care,39.60,15.573
ABBV,Health Care,53.95,2.954
ACN,Information Technology,79.79,8.326
ADBE,Information Technology,64.30,13.262
AES,Utilities,13.61,5.781


строки, где 6 < Price < 10, только столбец Price

In [78]:
sp500.loc[(sp500['Price'] < 10) & (sp500['Price'] > 6), 'Price']

Symbol
HCBK    9.80
HBAN    9.10
SLM     8.82
WIN     9.38
Name: Price, dtype: float64

сектор Health Care и Price ≥ 100

In [79]:
sp500.loc[(sp500.Sector == 'Health Care') & (sp500.Price >= 100), ['Price', 'Sector']].head()

,Price,Sector
Symbol,,
ACT,213.77,Health Care
ALXN,162.30,Health Care
AGN,166.92,Health Care
AMGN,114.33,Health Care
BCR,146.62,Health Care


### 8.3 isin()

In [80]:
mask = sp500.Sector.isin(['Information Technology', 'Financials'])
print('отобрано строк:', mask.sum())
sp500[mask].head()

отобрано строк: 146


,Sector,Price,Book Value
Symbol,,,
ACN,Information Technology,79.79,8.326
ACE,Financials,102.91,86.897
ADBE,Information Technology,64.30,13.262
AFL,Financials,61.31,34.527
AKAM,Information Technology,53.65,15.193


отрицание `~`: все секторы, **кроме** перечисленных

In [81]:
sp500[~mask].Sector.unique()

<StringArray>
[                'Industrials',                 'Health Care',
                   'Utilities',                   'Materials',
            'Consumer Staples',      'Consumer Discretionary',
                      'Energy', 'Telecommunications Services']
Length: 8, dtype: str

### 8.4 query()
Тот же фильтр, что в 8.2, но короче. Внешняя переменная — через `@`.

In [82]:
min_price = 100
sp500.query("Sector == 'Health Care' and Price >= @min_price")[['Price', 'Sector']].head()

,Price,Sector
Symbol,,
ACT,213.77,Health Care
ALXN,162.30,Health Care
AGN,166.92,Health Care
AMGN,114.33,Health Care
BCR,146.62,Health Care


имена столбцов с пробелами в `query` заключаются в обратные кавычки

In [83]:
sp500.query('`Book Value` > Price').head()

,Sector,Price,Book Value
Symbol,,,
AIG,Financials,53.60,71.7750
AIZ,Financials,66.57,70.9100
BAC,Financials,14.72,20.7540
BRK-B,Financials,126.80,138425.4531
C,Financials,47.29,66.2470


## 9. Сортировка

### Теория
- `sort_values(by, ascending=True)` — по значениям одного или нескольких столбцов; для нескольких столбцов `ascending` можно задать списком.
- `sort_index(axis=0)` — по меткам строк; `axis=1` — по именам столбцов.
- `nlargest(n, col)` / `nsmallest(n, col)` — n наибольших / наименьших значений (быстрее полной сортировки).
- Методы возвращают **новый** объект; исходный не меняется.

In [84]:
sp500.sort_index().head()

,Sector,Price,Book Value
Symbol,,,
A,Health Care,56.18,16.928
AA,Materials,13.52,9.670
AAPL,Information Technology,614.13,139.460
ABBV,Health Care,53.95,2.954
ABC,Health Care,71.64,9.430


In [85]:
sp500.sort_index(axis=1).head()

,Book Value,Price,Sector
Symbol,,,
MMM,26.668,141.14,Industrials
ABT,15.573,39.60,Health Care
ABBV,2.954,53.95,Health Care
ACN,8.326,79.79,Information Technology
ACE,86.897,102.91,Financials


In [86]:
sp500.sort_values(by='Price', ascending=False).head()

,Sector,Price,Book Value
Symbol,,,
PCLN,Industrials,1197.12,137.886
GHC,Consumer Discretionary,677.29,0.000
AAPL,Information Technology,614.13,139.460
GOOG,Information Technology,552.70,135.977
AZO,Consumer Discretionary,540.90,-51.275


сортировка по нескольким столбцам: сектор по алфавиту, внутри сектора — по убыванию цены

In [87]:
sp500.sort_values(by=['Sector', 'Price'], ascending=[True, False]).head(20)

,Sector,Price,Book Value
Symbol,,,
GHC,Consumer Discretionary,677.29,0.000
AZO,Consumer Discretionary,540.90,-51.275
CMG,Consumer Discretionary,522.32,52.915
AMZN,Consumer Discretionary,312.24,22.452
WYNN,Consumer Discretionary,207.86,-0.719
...,...,...,...
FOSL,Consumer Discretionary,101.72,18.980
TIF,Consumer Discretionary,96.59,22.187
KORS,Consumer Discretionary,96.40,7.992


In [88]:
sp500.nlargest(5, 'Price')

,Sector,Price,Book Value
Symbol,,,
PCLN,Industrials,1197.12,137.886
GHC,Consumer Discretionary,677.29,0.000
AAPL,Information Technology,614.13,139.460
GOOG,Information Technology,552.70,135.977
AZO,Consumer Discretionary,540.90,-51.275


In [89]:
sp500.nsmallest(5, 'Price')

,Sector,Price,Book Value
Symbol,,,
BEAM,Consumer Discretionary,0.00,NaN
FTR,Telecommunications Services,5.81,3.989
SLM,Financials,8.82,11.895
HBAN,Financials,9.10,6.995
WIN,Telecommunications Services,9.38,1.199


## 10. Изменение структуры: создание, переименование, удаление столбцов

### Теория
**Создание столбцов**
- `df['new'] = выражение` — добавить в конец (изменяет `df`);
- `df.insert(позиция, 'new', значения)` — вставить на нужную позицию (изменяет `df`);
- `df.assign(new=выражение, ...)` — вернуть **новый** датафрейм; можно использовать `lambda` для ссылки на только что созданные столбцы.

**Переименование**
- `df.rename(columns={'старое': 'новое'})` / `rename(index={...})` — возвращает новый объект;
- `df.columns = [...]` — заменить все имена сразу.

**Удаление**
- `df.drop(columns=[...])` / `df.drop(index=[...])` — возвращает новый объект (то же, что `axis=1` / `axis=0`);
- `del df['col']` — удалить столбец на месте;
- `df.pop('col')` — удалить столбец на месте и вернуть его как Series.

Параметр `inplace=True` изменяет объект на месте, но его использование не рекомендуется: присваивание результата (`df = df.drop(...)`) понятнее.

### 10.1 Создание

In [90]:
sp500_copy = sp500.copy()
sp500_copy['RoundedPrice'] = sp500_copy.Price.round()
sp500_copy.head(3)

,Sector,Price,Book Value,RoundedPrice
Symbol,,,,
MMM,Industrials,141.14,26.668,141.0
ABT,Health Care,39.60,15.573,40.0
ABBV,Health Care,53.95,2.954,54.0


In [91]:
sp500_copy = sp500.copy()
sp500_copy.insert(1, 'RoundedPrice', sp500_copy.Price.round())
sp500_copy.head(3)

,Sector,RoundedPrice,Price,Book Value
Symbol,,,,
MMM,Industrials,141.0,141.14,26.668
ABT,Health Care,40.0,39.60,15.573
ABBV,Health Care,54.0,53.95,2.954


`assign` создаёт два столбца, второй вычисляется из первого через `lambda`

In [92]:
sp500.assign(Rounded_Price=sp500.Price.round(),
             R_BookValue_Price=lambda x: x['Book Value'] / x['Rounded_Price']).head(3)

,Sector,Price,Book Value,Rounded_Price,R_BookValue_Price
Symbol,,,,,
MMM,Industrials,141.14,26.668,141.0,0.189135
ABT,Health Care,39.60,15.573,40.0,0.389325
ABBV,Health Care,53.95,2.954,54.0,0.054704


### 10.2 Переименование

In [93]:
sp500_copy = sp500.rename(columns={'Book Value': 'BookValue'})
print('исходный:', sp500.columns.tolist())
print('копия   :', sp500_copy.columns.tolist())

исходный: ['Sector', 'Price', 'Book Value']
копия   : ['Sector', 'Price', 'BookValue']


In [94]:
sp500.rename(index={'MMM': '3M'}).head(2)

,Sector,Price,Book Value
Symbol,,,
3M,Industrials,141.14,26.668
ABT,Health Care,39.60,15.573


### 10.3 Удаление

`drop` — новый объект без столбца `Sector`; исходный не изменился

In [95]:
sp500_copy = sp500.copy()
display(sp500_copy.drop(columns=['Sector']).head(3))
print(sp500_copy.columns.tolist())

,Price,Book Value
Symbol,,
MMM,141.14,26.668
ABT,39.60,15.573
ABBV,53.95,2.954


['Sector', 'Price', 'Book Value']


удаление строк по меткам

In [96]:
sp500.iloc[:5].drop(index=['ABT', 'ACN'])

,Sector,Price,Book Value
Symbol,,,
MMM,Industrials,141.14,26.668
ABBV,Health Care,53.95,2.954
ACE,Financials,102.91,86.897


`del` и `pop` удаляют на месте; `pop` ещё и возвращает удалённый столбец. Для Series работают аналогично (по метке элемента).

In [97]:
sp500_copy = sp500.copy()
del sp500_copy['Price']
popped_column = sp500_copy.pop('Sector')
display(sp500_copy.head(3))
popped_column.head(3)

,Book Value
Symbol,
MMM,26.668
ABT,15.573
ABBV,2.954


Symbol
MMM     Industrials
ABT     Health Care
ABBV    Health Care
Name: Sector, dtype: str

In [98]:
Simpsons_copy = Simpsons.copy()
del Simpsons_copy['Maggie']
Simpsons_copy

First name
Homer    120
Marge     60
Bart      35
Lisa      30
Name: Simpsons weight, dtype: int64

## 11. Арифметические операции и выравнивание данных

### Теория
Арифметические операции (`+ - * / // % **`) в pandas **векторизованы** — применяются ко всем элементам сразу:
- столбец и скаляр: `df['Price'] * 1.1`;
- столбец и столбец: `df['Price'] / df['Book Value']`;
- DataFrame и Series: по умолчанию Series сопоставляется со **столбцами**.

**Выравнивание по индексу.** При операции над двумя объектами pandas сначала сопоставляет их **метки**, а не позиции. Для меток, которых нет хотя бы в одном объекте, результат — `NaN`. Чтобы заменить отсутствующие значения, используют методы `add`, `sub`, `mul`, `div` с параметром `fill_value`.

Агрегирующие функции: `sum()`, `mean()`, `median()`, `min()`, `max()`, `std()`, `count()`.

### 11.1 Векторная арифметика

In [137]:
ar = sp500.head().copy()
ar['Price_x2'] = ar['Price'] * 2
ar['Price_minus_BV'] = ar['Price'] - ar['Book Value']
ar['Price_to_Book'] = (ar['Price'] / ar['Book Value']).round(2)
ar

,Sector,Price,Book Value,Price_x2,Price_minus_BV,Price_to_Book
Symbol,,,,,,
MMM,Industrials,141.14,26.668,282.28,114.472,5.29
ABT,Health Care,39.60,15.573,79.20,24.027,2.54
ABBV,Health Care,53.95,2.954,107.90,50.996,18.26
ACN,Information Technology,79.79,8.326,159.58,71.464,9.58
ACE,Financials,102.91,86.897,205.82,16.013,1.18


операция DataFrame − Series: из каждого столбца вычитается его среднее

In [100]:
num = sp500[['Price', 'Book Value']]
(num - num.mean()).head()

,Price,Book Value
Symbol,,
MMM,63.7714,-274.943301
ABT,-37.7686,-286.038301
ABBV,-23.4186,-298.657301
ACN,2.4214,-293.285301
ACE,25.5414,-214.714301


In [101]:
print('сумма   :', round(sp500['Price'].sum(), 2))
print('среднее :', round(sp500['Price'].mean(), 2))
print('медиана :', sp500['Price'].median())
print('макс    :', sp500['Price'].max())



сумма   : 38684.3
среднее : 77.37
медиана : 58.355000000000004
макс    : 1197.12


### 11.2 Выравнивание Series

In [102]:
s_1 = pd.Series([77, 33, 11], index=['a', 'b', 'f'])
s_2 = pd.Series([11, 5, 6], index=['c', 'b', 'a'])
display(pd.DataFrame({'s_1': s_1, 's_2': s_2, 's_1 + s_2': s_1 + s_2,
                      'add(fill_value=0)': s_1.add(s_2, fill_value=0)}))

,s_1,s_2,s_1 + s_2,add(fill_value=0)
a,77.0,6.0,83.0,83.0
b,33.0,5.0,38.0,38.0
c,NaN,11.0,NaN,11.0
f,11.0,NaN,NaN,11.0


метки не обязательно уникальны: 3 метки 'a' и 2 метки 'a' дают 3×2 = 6 меток 'a' в результате

In [138]:
s_1 = pd.Series([77, 33, 15, 3], index=['a', 'a', 'a', 'd'])
s_2 = pd.Series([11, 5, 6], index=['c', 'a', 'a'])
s_2 + s_1

a    82.0
a    38.0
a    20.0
a    83.0
a    39.0
a    21.0
c     NaN
d     NaN
dtype: float64

### 11.3 Выравнивание DataFrame
Выравнивание происходит и по строкам, и по столбцам.

In [104]:
sp500_part_1 = sp500.iloc[0:5, 0:2].copy()
sp500_part_2 = sp500.iloc[2:7, 1:3].copy()
display(sp500_part_1, sp500_part_2)
sp500_part_1 + sp500_part_2

,Sector,Price
Symbol,,
MMM,Industrials,141.14
ABT,Health Care,39.60
ABBV,Health Care,53.95
ACN,Information Technology,79.79
ACE,Financials,102.91


,Price,Book Value
Symbol,,
ABBV,53.95,2.954
ACN,79.79,8.326
ACE,102.91,86.897
ACT,213.77,55.188
ADBE,64.30,13.262


,Book Value,Price,Sector
Symbol,,,
ABBV,NaN,107.90,NaN
ABT,NaN,NaN,NaN
ACE,NaN,205.82,NaN
ACN,NaN,159.58,NaN
ACT,NaN,NaN,NaN
ADBE,NaN,NaN,NaN
MMM,NaN,NaN,NaN


выравнивание происходит и при создании DataFrame из Series с разными индексами

In [105]:
df = pd.DataFrame({'col_1': pd.Series([70, 90]),
                   'col_2': pd.Series([71, 91]),
                   'col_3': pd.Series([85, 87], index=[1, 2])})
df

,col_1,col_2,col_3
0,70.0,71.0,NaN
1,90.0,91.0,85.0
2,NaN,NaN,87.0


## 12. Индексация подробнее

### Теория
Индекс — не просто «номера строк». Хорошо выбранный индекс:
- ускоряет поиск: `loc` по индексу использует хеш-таблицу, а фильтр по столбцу `df[df.key == x]` проверяет все строки;
- позволяет выравнивать данные при арифметике и объединении таблиц.

Операции с индексом:
- `set_index(col)` — сделать столбец индексом; `reset_index()` — вернуть индекс в столбцы;
- `reindex(index=..., columns=...)` — привести к заданному набору меток (новые метки получают `NaN` или `fill_value`);
- **MultiIndex** — иерархический индекс из нескольких уровней; выбор по уровню — `xs(метка, level=...)` или `loc[(уровень0, уровень1)]`.

### 12.1 Мотивация: скорость доступа

In [106]:
np.random.seed(123)
df = pd.DataFrame({'value': np.random.random(10000), 'key': range(100, 10100)})
df_with_index = df.set_index('key')
df_with_index.head()

,value
key,
100,0.696469
101,0.286139
102,0.226851
103,0.551315
104,0.719469


In [107]:
%timeit df[df.key == 10099]

301 μs ± 5.9 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [108]:
%timeit df_with_index.loc[10099]

29.9 μs ± 899 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


<center><font color='green'><b>Вывод: поиск по индексу выполняется в десятки раз быстрее, чем фильтрация по столбцу.</b></font></center>

### 12.2 reset_index и set_index

In [109]:
index_moved_to_col = sp500.reset_index()
index_moved_to_col.head(3)

,Symbol,Sector,Price,Book Value
0,MMM,Industrials,141.14,26.668
1,ABT,Health Care,39.60,15.573
2,ABBV,Health Care,53.95,2.954


In [110]:
index_moved_to_col.set_index('Sector').head(3)

,Symbol,Price,Book Value
Sector,,,
Industrials,MMM,141.14,26.668
Health Care,ABT,39.60,15.573
Health Care,ABBV,53.95,2.954


### 12.3 Переиндексация (reindex)

`loc` с несуществующей меткой вызывает ошибку, а `reindex` создаёт строку с `NaN`

In [111]:
try:
    sp500.loc[['MMM', 'ABBV', 'NEW VALUE']]
except KeyError as e:
    print('KeyError:', e)
sp500.reindex(index=['MMM', 'ABBV', 'NEW VALUE'])

KeyError: "['NEW VALUE'] not in index"


,Sector,Price,Book Value
Symbol,,,
MMM,Industrials,141.14,26.668
ABBV,Health Care,53.95,2.954
NEW VALUE,NaN,NaN,NaN


In [112]:
sp500.reindex(columns=['Price', 'Book Value', 'NewCol'], fill_value=0).head(3)

,Price,Book Value,NewCol
Symbol,,,
MMM,141.14,26.668,0
ABT,39.60,15.573,0
ABBV,53.95,2.954,0


### 12.4 Операции над множествами индексов
Какие компании попали в обе случайные выборки, в любую из них, только в первую?

In [113]:
part_1 = sp500.sample(100, random_state=333)
part_2 = sp500.sample(100, random_state=777)
print('объединение:', len(part_1.index.union(part_2.index)))
print('пересечение:', len(part_1.index.intersection(part_2.index)))
print('разность   :', len(part_1.index.difference(part_2.index)))

объединение: 177
пересечение: 23
разность   : 77


### 12.5 Иерархический индекс (MultiIndex)

In [114]:
multi_fi = sp500.reset_index().set_index(['Sector', 'Symbol']).sort_index()
print(type(multi_fi.index), '| уровней:', multi_fi.index.nlevels)
multi_fi.head()

<class 'pandas.MultiIndex'> | уровней: 2


Price  Book Value
Sector                 Symbol                    
Consumer Discretionary AMZN    312.24      22.452
                       AN       56.38      17.327
                       AZO     540.90     -51.275
                       BBBY     61.36      19.563
                       BBY      27.01      12.735

каждый уровень — отдельный Index

In [115]:
multi_fi.index.levels[0]

Index(['Consumer Discretionary', 'Consumer Staples', 'Energy', 'Financials',
       'Health Care', 'Industrials', 'Information Technology', 'Materials',
       'Telecommunications Services', 'Utilities'],
      dtype='str', name='Sector')

In [116]:
multi_fi.reorder_levels([1, 0], axis=0).head()

,,Price,Book Value
Symbol,Sector,,
AMZN,Consumer Discretionary,312.24,22.452
AN,Consumer Discretionary,56.38,17.327
AZO,Consumer Discretionary,540.90,-51.275
BBBY,Consumer Discretionary,61.36,19.563
BBY,Consumer Discretionary,27.01,12.735


все акции сектора Industrials (уровень 0 исчезает из результата)

In [117]:
multi_fi.xs('Industrials').head()

,Price,Book Value
Symbol,,
ADT,32.89,17.627
ALLE,52.46,0.000
APH,95.71,18.315
AVY,48.20,15.616
BA,132.41,19.870


отбор по уровню 1 (Symbol)

In [118]:
multi_fi.xs('ALLE', level=1)

,Price,Book Value
Sector,,
Industrials,52.46,0.0


комбинация уровней через кортеж

In [119]:
multi_fi.loc[('Industrials', 'UPS')]

Price         102.73
Book Value      6.79
Name: (Industrials, UPS), dtype: float64

## 13. Работа с большим датасетом: память, типы данных, usecols, dtype, chunksize

### Теория
На маленьких таблицах эффективность незаметна, но при сотнях тысяч и миллионах строк данные могут занять гигабайты памяти.

**Сколько занимает DataFrame:** примерно `число строк × сумма размеров значений всех столбцов`.

| Тип | Байт на значение | Диапазон / комментарий |
|---|---|---|
| `int8` / `int16` / `int32` / `int64` | 1 / 2 / 4 / 8 | int8: −128…127 |
| `float32` / `float64` | 4 / 8 | float32 ≈ 7 значащих цифр |
| `bool` | 1 | |
| `datetime64` | 8 | |
| `object` / `str` (строки) | десятки байт | каждая строка — отдельный объект Python |
| `category` | 1–2 (коды) + словарь значений | для повторяющихся строк |

**Способы уменьшить память:**
1. `memory_usage(deep=True)` / `info(memory_usage='deep')` — измерить реальный расход (без `deep` строки не учитываются).
2. **Подобрать типы**: `astype()`, `pd.to_numeric(..., downcast=...)`, `category` для строк с малым числом уникальных значений.
3. **`usecols`** — не загружать ненужные столбцы.
4. **`dtype={...}`** — задать компактные типы уже при чтении.
5. **Фильтрация строк и выбор столбцов** — уменьшают соответственно число строк и число столбцов в формуле выше. После фильтрации делайте `.copy()` и удаляйте большой объект `del df`.
6. **`chunksize`** — обрабатывать файл по частям, если он не помещается в память.

### 13.1 Генерация датасета на 1 000 000 строк

In [120]:
N = 1_000_000
rng = np.random.default_rng(0)
big = pd.DataFrame({
    'order_id': np.arange(N),
    'date': pd.date_range('2023-01-01', periods=N, freq='min').strftime('%Y-%m-%d'),
    'region': rng.choice(['Центр', 'Юг', 'Север', 'Урал', 'Сибирь'], N),
    'category': rng.choice(['Электроника', 'Одежда', 'Продукты', 'Книги'], N),
    'qty': rng.integers(1, 20, N),
    'price': rng.uniform(10, 5000, N).round(2),
    'comment': rng.choice(['ok', 'возврат', 'доставка задержана', ''], N)})
big.to_csv('sales_1m.csv', index=False)
del big
print(f'Размер файла: {os.path.getsize("sales_1m.csv") / 1024**2:.1f} МБ')

Размер файла: 65.5 МБ


### 13.2 Полная загрузка и измерение памяти

In [121]:
def mb(d):
    """Объём DataFrame в памяти, МБ (с учётом строк)."""
    return d.memory_usage(deep=True).sum() / 1024**2

t = time.time()
sales = pd.read_csv('sales_1m.csv')
t_full = time.time() - t
print(f'Чтение: {t_full:.2f} c, shape = {sales.shape}')
sales.info(memory_usage='deep')

Чтение: 1.76 c, shape = (1000000, 7)
<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 7 columns):
 #   Column    Non-Null Count    Dtype  
---  ------    --------------    -----  
 0   order_id  1000000 non-null  int64  
 1   date      1000000 non-null  str    
 2   region    1000000 non-null  str    
 3   category  1000000 non-null  str    
 4   qty       1000000 non-null  int64  
 5   price     1000000 non-null  float64
 6   comment   750344 non-null   str    
dtypes: float64(1), int64(2), str(4)
memory usage: 290.9 MB


память по столбцам: строковые столбцы занимают больше всего

In [122]:
mem = (sales.memory_usage(deep=True) / 1024**2).round(2)
print(mem)
print(f'Итого: {mem.sum():.1f} МБ')

Index        0.00
order_id     7.63
date        56.27
region      71.34
category    77.25
qty          7.63
price        7.63
comment     63.21
dtype: float64
Итого: 291.0 МБ


### 13.3 Оптимизация типов данных

In [123]:
opt = sales.copy()
opt['order_id'] = pd.to_numeric(opt['order_id'], downcast='integer')
opt['qty'] = opt['qty'].astype('int8')
opt['price'] = opt['price'].astype('float32')
for col in ['region', 'category', 'comment']:
    opt[col] = opt[col].astype('category')
opt['date'] = pd.to_datetime(opt['date'])
print(f'До: {mb(sales):.1f} МБ, после: {mb(opt):.1f} МБ, экономия в {mb(sales) / mb(opt):.1f} раз')
opt.dtypes

До: 315.6 МБ, после: 19.1 МБ, экономия в 16.5 раз


order_id             int32
date        datetime64[us]
region            category
category          category
qty                   int8
price              float32
comment           category
dtype: object

почему `category` так эффективен: хранится словарь из нескольких уникальных значений и маленький код для каждой строки

In [124]:
print(opt['region'].cat.categories)
print(opt['region'].cat.codes.dtype, opt['region'].cat.codes.head().tolist())

Index(['Север', 'Сибирь', 'Урал', 'Центр', 'Юг'], dtype='str')
int8 [1, 2, 0, 4, 4]


### 13.4 usecols и dtype при чтении

In [125]:
t = time.time()
part = pd.read_csv('sales_1m.csv', usecols=['region', 'qty', 'price'],
                   dtype={'region': 'category', 'qty': 'int8', 'price': 'float32'})
t_part = time.time() - t
print(f'Все столбцы     : {mb(sales):7.1f} МБ, {t_full:.2f} c')
print(f'usecols + dtype : {mb(part):7.1f} МБ, {t_part:.2f} c')
part.dtypes

Все столбцы     :   315.6 МБ, 1.76 c
usecols + dtype :     5.7 МБ, 1.35 c


region    category
qty           int8
price      float32
dtype: object

### 13.5 Почему фильтрация и выбор столбцов уменьшают объём данных

In [126]:
cond = (sales['region'] == 'Юг') & (sales['price'] > 4000)
cols = sales[['region', 'price']]
filtered = sales[cond].copy()
both = sales.loc[cond, ['region', 'price']].copy()
report = pd.DataFrame({
    'строк': [len(sales), len(cols), len(filtered), len(both)],
    'столбцов': [sales.shape[1], cols.shape[1], filtered.shape[1], both.shape[1]],
    'МБ': [mb(sales), mb(cols), mb(filtered), mb(both)]},
    index=['весь датасет', 'только 2 столбца', 'фильтр строк', 'фильтр + 2 столбца']).round(2)
report['доля памяти, %'] = (report['МБ'] / report['МБ'].iloc[0] * 100).round(1)
report

,строк,столбцов,МБ,"доля памяти, %"
весь датасет,1000000,7,315.55,100.0
только 2 столбца,1000000,2,88.32,28.0
фильтр строк,39979,7,12.55,4.0
фильтр + 2 столбца,39979,2,3.47,1.1


### 13.6 chunksize: обработка файла по частям
В памяти одновременно находится только один кусок; результаты по кускам объединяются.

In [127]:
total = None
max_chunk_mb = 0
for chunk in pd.read_csv('sales_1m.csv', usecols=['region', 'qty', 'price'],
                         dtype={'region': 'category', 'qty': 'int8', 'price': 'float32'},
                         chunksize=100_000):
    max_chunk_mb = max(max_chunk_mb, mb(chunk))
    chunk['revenue'] = chunk['qty'] * chunk['price']
    part_sum = chunk.groupby('region', observed=True)['revenue'].sum() #groupby группирует по региону и будет подробнее рассмотрен в следующей работе
    total = part_sum if total is None else total.add(part_sum, fill_value=0)
print(f'Макс. память одного куска: {max_chunk_mb:.1f} МБ')
total.sort_values(ascending=False).round(0).astype('int64')

Макс. память одного куска: 0.6 МБ


region
Север     5022219264
Сибирь    5017728512
Урал      5008742912
Центр     5005609984
Юг        4995490304
Name: revenue, dtype: int64

проверка: результат по частям совпадает с расчётом на всём датасете

In [128]:
check = (sales['qty'] * sales['price']).groupby(sales['region']).sum()
print(np.allclose(check.sort_index().values, total.sort_index().values, rtol=1e-4))
del sales, opt, part, cols, filtered, both

True


### Выводы
1. Строковые столбцы `object`/`str` — главный потребитель памяти; `category` сокращает их в десятки раз.
2. `usecols` и `dtype` уменьшают память и ускоряют чтение уже на этапе загрузки.
3. Выбор столбцов и фильтрация строк уменьшают объём данных пропорционально оставшейся части.
4. `chunksize` позволяет обрабатывать файлы больше оперативной памяти.

## 14. Задания для самостоятельной работы

Все задания выполняются только средствами, разобранными в этой работе; у каждого указаны разделы, на которые оно опирается. Решения записывайте в ячейки под условием. Ответы на вопросы «объясните» оформляйте в markdown-ячейках.

Данные: `sp500.csv` (переменная `DATA_PATH`) и `sales_1m.csv` (создаётся в разделе 13).

### Задание 1. Создание структур данных и Index
*Разделы 2–3*
1. Создайте Series `grades` с оценками пяти студентов из **словаря** (имя → оценка). Задайте Series имя `'grade'`, а индексу — имя `'student'`.
2. Создайте DataFrame `students` из **словаря списков** со столбцами `group` и `age`, указав в качестве `index` имена тех же студентов.
3. Создайте DataFrame `scores` из **массива NumPy**: зафиксируйте `np.random.seed(1)`, сгенерируйте `np.random.normal(size=15)`, преобразуйте в форму 5×3 через `reshape`, задайте столбцы `test_1`, `test_2`, `test_3` и тот же индекс.
4. Создайте Series `weeks` из `np.arange(1, 11)` и Series `points` из `np.linspace(0, 100, 5)`.
5. Создайте `pd.Index` из трёх имён (два из них должны быть в `students`, одно — нет). Найдите объединение, пересечение и разность с `students.index`.
6. Попробуйте заменить первый элемент индекса `students.index[0] = 'X'`, перехватив ошибку через `try/except`. Объясните, почему возникает ошибка.

In [129]:
slov_1 = {"Asan": 2, "nikita": 4, "Artem": 5, "Vasya": 3, "Valera": 4}
grades = pd.Series(slov_1)
grades.name = "grade"; grades.index.name = "student"
print(grades)

students_dict = {
    'group': ['A', 'B', 'A', 'B', 'A'],
    'age': [19, 20, 18, 21, 19]
}
students = pd.DataFrame(students_dict,index = grades.index)
print(students)
rng = np.random.seed(1)
scores = pd.DataFrame(np.random.normal(size = 15, loc = 10, scale = 3).reshape(5,3), columns = ["test_1", "test_2", "test_3"], index = grades.index)
print(scores)

weeks = pd.Series(np.arange(1,11))
points = pd.Series(np.linspace(0,100,5))
student_new = pd.Index(['Asan', 'Artem', 'Олег'])

print("Объединение:", list(student_new.union(students.index)))
print("Пересечение:", list(student_new.intersection(students.index)))
print("Разность, новые без старых:", list(student_new.difference(students.index)))
try:
    students.index[0] = 'X'
except Exception as e:
    print(f"поймана ошибка: {e}  нельзя менять объект 'индекс'")

student
Asan      2
nikita    4
Artem     5
Vasya     3
Valera    4
Name: grade, dtype: int64
        group  age
student           
Asan        A   19
nikita      B   20
Artem       A   18
Vasya       B   21
Valera      A   19
            test_1     test_2     test_3
student                                 
Asan     14.873036   8.164731   8.415485
nikita    6.781094  12.596223   3.095384
Artem    15.234435   7.716379  10.957117
Vasya     9.251889  14.386324   3.819578
Valera    9.032748   8.847837  13.401308
Объединение: ['Artem', 'Asan', 'Valera', 'Vasya', 'nikita', 'Олег']
Пересечение: ['Asan', 'Artem']
Разность, новые без старых: ['Олег']
поймана ошибка: Index does not support mutable operations  нельзя менять объект 'индекс'


### Задание 2. Чтение файлов и первичный осмотр
*Разделы 4–5*
1. Прочитайте первые 5 строк `sp500.csv` (`nrows`), а затем весь файл в DataFrame `sp` только со столбцами `Symbol`, `Name`, `Sector`, `Price`, `Dividend Yield`, `Market Cap`, сделав `Symbol` индексом.
2. Выведите `head(3)`, `tail(3)`, `shape`, `columns`, `dtypes`.
3. Выведите `info()` и по столбцу `Non-Null Count` определите, в каких столбцах есть пропуски и сколько их (сравните с `len(sp)`).
4. Выведите `describe()`. Найдите в нём медиану цены (строка `50%`) и проверьте её методом `median()`.
5. Очистите названия секторов так, как показано в разделе 5 (`str.strip()` и `replace()`), и выведите число уникальных секторов `nunique()`.
6. Сохраните первые 20 строк в Excel и JSON (`orient='records'`), прочитайте обратно функциями `read_excel()` и `read_json()` и проверьте совпадение методом `equals()`.

In [130]:

print(pd.read_csv("data/sp500.csv", nrows = 5))
sp = pd.read_csv("data/sp500.csv", usecols = ['Symbol', 'Name', 'Sector', 'Price', 'Dividend Yield', 'Market Cap'], index_col = 'Symbol')

print(sp.head(3))
print(sp.tail(3))
print(sp.shape)
print(sp.columns)
print(sp.dtypes)

print(len(sp) - sp.count())

print(sp.describe())

median_from_describe = sp['Price'].describe()['50%']
median_method = sp['Price'].median()
print(f"\nМедиана из describe: {median_from_describe}")
print(f"Медиана через метод median(): {median_method}")
print(sp)

print("\nУникальные секторы до очистки:", sp['Sector'].nunique())
print(sp['Sector'].unique())
sp['Sector'] = sp['Sector'].str.strip().replace({'Industries': 'Industrials'})
print("Уникальные секторы после очистки:", sp['Sector'].nunique())
print(sp['Sector'].unique())

sp_20 = sp.head(20).reset_index()
sp_20.to_excel('sp500_20.xlsx', index=False, sheet_name='data')
sp_20.to_json('sp500_20.json', orient='records', force_ascii=False)

df_excel = pd.read_excel('sp500_20.xlsx', index_col='Symbol')
df_json = pd.read_json('sp500_20.json', orient='records').set_index('Symbol')

print("\nСовпадение Excel и исходника:", sp.head(20).equals(df_excel))
print("Совпадение JSON и исходника:", sp.head(20).equals(df_json))


  Symbol                 Name                  Sector   Price  Dividend Yield  \
0    MMM               3M Co.             Industrials  141.14            2.12   
1    ABT  Abbott Laboratories             Health Care   39.60            1.82   
2   ABBV          AbbVie Inc.             Health Care   53.95            3.02   
3    ACN            Accenture  Information Technology   79.79            2.34   
4    ACE          ACE Limited              Financials  102.91            2.21   

   Price/Earnings  Earnings/Share  Book Value  52 week low  52 week high  \
0           20.33           6.900      26.668       107.15        143.37   
1           25.93           1.529      15.573        32.70         40.49   
2           20.87           2.570       2.954        40.10         54.78   
3           19.53           4.068       8.326        69.00         85.88   
4           10.00          10.293      86.897        84.73        104.07   

   Market Cap  EBITDA  Price/Sales  Price/Book  \
0     

### Задание 3. Доступ к данным
*Раздел 6*
На датафрейме `sp` из задания 2:
1. Выведите цену компании `MSFT` четырьмя способами: `loc`, `at`, `iloc` и `iat` (позицию строки найдите через `index.get_loc()`, позицию столбца — через `sp.columns.get_loc()`: столбцы тоже являются объектом Index, раздел 3).
2. С помощью `type()` определите тип результата для `sp.loc['MSFT', 'Price']`, `sp.loc['MSFT']`, `sp.loc[['MSFT'], ['Price']]`. Сопоставьте с таблицей раздела 6.
3. С помощью `iloc` выведите строки с 10-й по 19-ю позицию и столбцы с 1-го по 3-й.
4. С помощью `loc` выведите строки от метки `ABT` до метки `ACN` включительно и только столбцы `Name` и `Price`.
5. Выведите последние 5 строк в обратном порядке срезом `iloc` с отрицательным шагом.
6. Объясните, почему `sp.iloc[0:3]` возвращает 3 строки, а `sp.loc['ABT':'ACN']` включает строку `ACN`.

In [131]:
print(sp.loc['MSFT']['Price']) # или [["MSFT"],['Price']]
print(sp.at["MSFT",'Price'])
index = sp.index.get_loc('MSFT')
col_price = sp.columns.get_loc('Price')
print(sp.iloc[index]['Price'])
print(sp.iat[index,col_price])

print(type(sp.loc["MSFT", "Price"]))
print(type(sp.loc["MSFT"]))
print(type(sp.loc[["MSFT"],['Price']]))

print(sp.iloc[10:20,1:4])
print(sp.loc["ABT": "ACN",["Name", "Price"]])

print(sp.iloc[-1:-6:-1])

#6. потомучто loc  включает правуб границу, а iloc не включает



40.12
40.12
40.12
40.12
<class 'numpy.float64'>
<class 'pandas.Series'>
<class 'pandas.DataFrame'>
                        Sector   Price  Dividend Yield
Symbol                                                
A                  Health Care   56.18            0.91
GAS                  Utilities   52.98            3.61
APD                  Materials  118.74            2.45
ARG                  Materials  105.66            1.83
AKAM    Information Technology   53.65             NaN
AA                   Materials   13.52            0.91
ALXN               Health Care  162.30             NaN
ATI                  Materials   40.99            1.33
ALLE               Industrials   52.46            0.16
AGN                Health Care  166.92            0.12
                       Name  Price
Symbol                            
ABT     Abbott Laboratories  39.60
ABBV            AbbVie Inc.  53.95
ACN               Accenture  79.79
                   Name                  Sector   Price  Dividend 

### Задание 4. Фильтрация
*Раздел 8*
1. Сохраните среднюю цену всех компаний в переменную `avg_price`. С помощью логической маски с `&` найдите компании сектора `Information Technology` с ценой выше `avg_price`. Сколько их (`sum()` маски или `len()` результата)?
2. Выполните тот же отбор через `query()`, передав среднюю цену как `@avg_price`. Убедитесь, что результаты совпадают (`equals()`).
3. С помощью `isin()` выберите компании секторов `Energy`, `Utilities`, `Materials`. Сколько их всего? Сколько в каждом из трёх секторов (отдельная маска `==` для каждого)?
4. С помощью `~` и `isin()` выберите компании, **не** относящиеся к секторам `Financials` и `Health Care`. Сколько их?
5. Ответьте с помощью `all()`, `any()`, `sum()`: все ли цены больше 0? есть ли компании с ценой выше 500? у скольких компаний `Dividend Yield` больше 3?
6. С помощью `where()` замените в столбце `Price` все цены меньше 50 на 0 и выведите первые 10 значений.

In [132]:
import numpy as np
price = np.mean(sp["Price"])
print("редняя цена: ",price)
mask = (sp['Price'] > price) & (sp["Sector"] == "Information Technology")
print("кол - во подходящих строк: ", sum(mask)) #sum т.к маска состоить из true false, в сумме их 500 шт(как и строк)

res_query = sp.query("Sector == 'Information Technology' and Price > @price")
print(len(res_query))
print("совпадение: ", sp[mask].equals(res_query))

print("выбранные секторы, кол-во: ",sum(sp["Sector"].isin(["Energy", "Utilities", "Materials"])))
print("Energy:", (sp['Sector'] == 'Energy').sum())
print("Utilities:", (sp['Sector'] == 'Utilities').sum())
print("Materials:", (sp['Sector'] == 'Materials').sum())

print("секторы кроме выбранных:", sum(sp["Sector"].isin(['Financials', 'Health Care'])))

print("все цены больше нуля? :", (sp["Price"]>0).all())
#print((sp["Price"]<=0).sum())
print("Есть цена > 500:", (sp['Price'] > 500).any())
print("дивиденды больше 3: ",(sp["Dividend Yield"] > 3).sum())

print((sp['Price'].where(sp["Price"] >= 50, 0)).head(10))



редняя цена:  77.3686
кол - во подходящих строк:  13
13
совпадение:  True
выбранные секторы, кол-во:  103
Energy: 41
Utilities: 33
Materials: 29
секторы кроме выбранных: 136
все цены больше нуля? : False
Есть цена > 500: True
дивиденды больше 3:  91
Symbol
MMM     141.14
ABT       0.00
ABBV     53.95
ACN      79.79
ACE     102.91
ACT     213.77
ADBE     64.30
AES       0.00
AET      76.39
AFL      61.31
Name: Price, dtype: float64


### Задание 5. Сортировка и изменение структуры
*Разделы 9–10*
Работайте с копией: `sp2 = sp.copy()`.
1. С помощью `assign()` добавьте столбец `Dividend` = `Price * Dividend Yield / 100` (дивиденд на акцию).
2. Оператором `[]` добавьте логический столбец `Is_Expensive` = цена выше медианной цены.
3. Методом `insert()` вставьте на позицию 1 столбец `Price_rounded` с округлённой ценой.
4. Переименуйте `Dividend Yield` в `DivYield`, `Market Cap` — в `MarketCap`. Удалите столбец `Name` методом `drop()`. Извлеките столбец `Is_Expensive` методом `pop()` и посчитайте, сколько в нём значений `True`.
5. Отсортируйте `sp2` по `Sector` по алфавиту, а внутри сектора — по `Dividend` по убыванию; выведите первые 10 строк.
6. Выведите 3 компании с наибольшим `Dividend` (`nlargest()`) и 3 самые дешёвые акции (`nsmallest()`). Отсортируйте `sp2` по индексу в обратном порядке (`sort_index()`).

In [133]:
#sp2 = sp.assign (Dividend = sp["Price"] * sp["Dividend Yield"]/100)
print(sp.columns)
sp2 = sp.assign(Dividend = lambda x : x['Price'] * x['Dividend Yield'] / 100)

sp2["Is_Expensive"] = sp2["Price"] > np.median(sp["Price"])
print((sp2["Is_Expensive"]).head())

sp2.insert(1,"Price_rounded", sp2["Price"].round())
print((sp2["Price_rounded"]).head())

sp2 = sp2.rename(columns = {"Dividend Yield": "DivYield", "Market Cap": "MarketCap"})
sp2 = sp2.drop(columns = "Name")
print(sp2.columns)

pop_Is = sp2.pop("Is_Expensive")
print(pop_Is.sum())

sp2_sort = sp2.sort_values(by = ["Sector", "Dividend"], ascending = [True, False])
print(sp2_sort.head(10))

print(sp2.nlargest(3, 'Dividend'))
print(sp2.nsmallest(3,'Price'))

sp2 = sp2.sort_index(ascending = False)
print(sp2.head())

Index(['Name', 'Sector', 'Price', 'Dividend Yield', 'Market Cap'], dtype='str')
Symbol
MMM      True
ABT     False
ABBV    False
ACN      True
ACE      True
Name: Is_Expensive, dtype: bool
Symbol
MMM     141.0
ABT      40.0
ABBV     54.0
ACN      80.0
ACE     103.0
Name: Price_rounded, dtype: float64
Index(['Price_rounded', 'Sector', 'Price', 'DivYield', 'MarketCap', 'Dividend',
       'Is_Expensive'],
      dtype='str')
250
        Price_rounded                  Sector   Price  DivYield  MarketCap  \
Symbol                                                                       
WYNN            208.0  Consumer Discretionary  207.86      3.60     20.957   
GHC             677.0  Consumer Discretionary  677.29      0.76        NaN   
MCD             102.0  Consumer Discretionary  102.00      3.08    100.800   
WHR             147.0  Consumer Discretionary  147.26      1.80     11.448   
HOT              79.0  Consumer Discretionary   78.73      3.00     15.116   
LB               57.0  Co

### Задание 6. Арифметика и выравнивание
*Разделы 11–12*
1. Создайте Series продаж по магазинам: `jan` с индексом `['A', 'B', 'C', 'D']` и `feb` с индексом `['B', 'C', 'D', 'E']`.
2. Посчитайте сумму продаж за два месяца оператором `+` и методом `add(fill_value=0)`. Объясните, откуда берутся `NaN` в первом случае.
3. Найдите магазины, присутствующие в обоих месяцах (`index.intersection()`), и для них посчитайте прирост продаж в процентах `(feb - jan) / jan * 100`, выбирая данные через `loc`.
4. Приведите `jan` и `feb` к общему индексу `['A', 'B', 'C', 'D', 'E']` методом `reindex(..., fill_value=0)` и сложите. Выведите этот результат и результат `add(fill_value=0)` рядом в одном DataFrame (как в разделе 11.2) и сравните значения.
5. Для `sp` из задания 2 одной операцией DataFrame − Series вычтите из столбцов `Price` и `Market Cap` их средние значения. Для `Price` найдите сумму, среднее и максимум.

In [153]:
jan = pd.Series([100, 200, 300, 400], index = ['A', 'B', 'C', 'D'])
feb = pd.Series([150, 250, 350, 450], index = ['B', 'C', 'D', 'E'])

sum_plus = jan + feb
print(sum_plus.tolist())
sum_add = jan.add(feb,fill_value = 0)
print(sum_add.tolist())

obsh = jan.index.intersection(feb.index)
print(obsh)
print("прирост:\n",(feb.loc[obsh] - jan.loc[obsh])/jan.loc[obsh] * 100)

idx = ['A', 'B', 'C', 'D','E']
jan_re = jan.reindex(index = idx)
feb_re = feb.reindex(index = idx)
sum_re = jan_re + feb_re
add_re = jan_re.add(feb_re, fill_value = 0)
display(pd.DataFrame({"jan": jan_re, "feb": feb_re,"+" : sum_re,'add': add_re }))

cop_sp = sp[["Price","Market Cap"]].copy() - sp[["Price","Market Cap"]].mean()
print(cop_sp.head())
print(f"sum: {cop_sp['Price'].sum()}\n"
      f"mean: {cop_sp['Price'].mean()}\n"
      f"max: {cop_sp['Price'].max()}")




[nan, 350.0, 550.0, 750.0, nan]
[100.0, 350.0, 550.0, 750.0, 450.0]
Index(['B', 'C', 'D'], dtype='str')
прирост:
 B   -25.000000
C   -16.666667
D   -12.500000
dtype: float64


,jan,feb,+,add
A,100.0,NaN,NaN,100.0
B,200.0,150.0,350.0,350.0
C,300.0,250.0,550.0,550.0
D,400.0,350.0,750.0,750.0
E,NaN,450.0,NaN,450.0


          Price  Market Cap
Symbol                     
MMM     63.7714   56.996475
ABT    -37.7686   24.128475
ABBV   -23.4186   50.435475
ACN      2.4214   15.164475
ACE     25.5414   -0.595525
sum: -1.0231815394945443e-12
mean: -2.0463630789890884e-15
max: 1119.7513999999999


### Задание 7. Большой датасет и экономия памяти
*Раздел 13*
Работайте с файлом `sales_1m.csv` (1 млн строк).
1. Прочитайте первые 1000 строк (`nrows`). По `dtypes`, `describe()` и `nunique()` решите, какой тип подходит каждому столбцу (`int8`, `int32`, `float32`, `category`). Составьте словарь `dtype`.
2. Загрузите файл: а) полностью без параметров; б) только со столбцами `date`, `category`, `qty`, `price` (`usecols`) и вашим `dtype`. Измерьте время чтения (`time.time()`) и память (`memory_usage(deep=True)`), оформите результат в таблицу DataFrame, как в разделе 13.5.
3. Из полной таблицы отберите заказы категории `'Книги'` с `qty > 10`: а) все столбцы; б) только `qty` и `price`. Для каждого варианта посчитайте число строк, память и долю памяти в процентах от полной таблицы. Объясните, почему память уменьшилась.
4. Не загружая весь файл, прочитайте его кусками по 200 000 строк (`chunksize`, только нужные столбцы и ваш `dtype`). В каждом куске отберите заказы категории `'Книги'` и прибавьте к трём счётчикам: число таких заказов (`len()`), сумму `qty` и сумму `price` (`sum()`). Также запомните максимальную память одного куска.
5. По итоговым счётчикам вычислите среднюю цену книги: **сумма цен / число заказов**. Проверьте результат, посчитав то же самое на полностью загруженных данных (`mean()` для отфильтрованного столбца).
6. Сделайте вывод (3–4 предложения): какие приёмы дали наибольшую экономию памяти и почему.

In [160]:
sales_1k = pd.read_csv('sales_1m.csv', nrows = 1000)
print(sales_1k.columns)
print(f"{sales_1k.dtypes}\n {sales_1k.describe()}\n {sales_1k.nunique()}")
print(sales_1k.sample(5))
typ = ['int32','datetime64','category','category','int8','float32','category']


Index(['order_id', 'date', 'region', 'category', 'qty', 'price', 'comment'], dtype='str')
order_id      int64
date            str
region          str
category        str
qty           int64
price       float64
comment         str
dtype: object
           order_id          qty        price
count  1000.000000  1000.000000  1000.000000
mean    499.500000     9.976000  2511.078000
std     288.819436     5.561326  1423.439834
min       0.000000     1.000000    21.210000
25%     249.750000     5.000000  1304.715000
50%     499.500000    10.000000  2530.350000
75%     749.250000    15.000000  3716.190000
max     999.000000    19.000000  4999.810000
 order_id    1000
date           1
region         5
category       4
qty           19
price        998
comment        3
dtype: int64
     order_id        date  region     category  qty    price  \
163       163  2023-01-01   Север     Продукты    1   841.06   
967       967  2023-01-01      Юг  Электроника    6  3451.29   
499       499  2023-01-01

## 15. Контрольные вопросы
1. Чем Series отличается от DataFrame? Что общего у них с массивом NumPy?
2. Какие свойства есть у объекта Index? Почему он неизменяемый?
3. В чём разница между `loc` и `iloc`? Как они обрабатывают правую границу среза?
4. Когда стоит использовать `at` и `iat` вместо `loc` и `iloc`?
5. Почему условия в логической маске нужно брать в скобки? Чем `&` отличается от `and`?
6. Что делают `isin()` и `query()`? Как передать в `query()` внешнюю переменную?
7. Чем `drop()` отличается от `del` и `pop()`?
8. Что такое выравнивание по индексу? Как избежать `NaN` при сложении Series с разными индексами?
9. Почему `memory_usage()` без `deep=True` занижает объём строковых столбцов?
10. Как `usecols`, `dtype`, фильтрация и `chunksize` помогают работать с большими файлами?